# Task 4: Style-Conditioned cGAN Validation & Qualitative Evaluation

Final evaluation of the trained `UNetGenerator` on the official FS2K test set as specified in the assignment.

### Mathematical Formulation & Evaluation Metrics
$$\hat{y} = G(x, s)$$
- **Pixel Reconstruction (L1)**: $\mathcal{L}_{L1}(y, \hat{y})$
- **Peak Signal-to-Noise Ratio (PSNR)**:
  $$\mathrm{PSNR} = 10 \log_{10}\left(\frac{1}{\mathrm{MSE}}\right)$$
- **Structural Similarity (SSIM)**
- **Style Conditioning Matrix**: 1 photo $\times$ 3 styles side-by-side comparison
- **Failure Mode Analysis**: Identification and diagnostic analysis of the 4 highest-error sketches (style bleeding, fine-edge blur, artifact formation).

In [ ]:
# 1. Imports
import sys
from pathlib import Path
import math
from collections import defaultdict
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESEARCH_ROOT = Path("..").resolve()
if str(RESEARCH_ROOT) not in sys.path:
    sys.path.insert(0, str(RESEARCH_ROOT))

from src.device_utils import get_device
from src.datasets import FS2KDataset
from src.models_gan import UNetGenerator
from src.checkpoint_utils import load_checkpoint
from src.losses import compute_ssim
from torch.utils.data import DataLoader

DEVICE = get_device()
STYLE_NAMES = ["Style 1", "Style 2", "Style 3"]

def psnr(pred, target, data_range=1.0):
    mse = F.mse_loss(pred.float(), target.float()).item()
    return 10.0 * math.log10((data_range**2) / max(mse, 1e-12))


In [ ]:
# 2. Configuration
GENERATOR_CKPT = (RESEARCH_ROOT / "checkpoints" / "task4_generator_final_best.pt"
                  if (RESEARCH_ROOT / "checkpoints" / "task4_generator_final_best.pt").exists()
                  else RESEARCH_ROOT / "checkpoints" / "task4_generator_best.pt")
BASE_CHANNELS  = 64
EMBED_DIM      = 8


## 1. Load Trained Generator

In [ ]:
# 3. Load generator — infer architecture from checkpoint
if not GENERATOR_CKPT.exists():
    raise FileNotFoundError(f"Train Task 4 first. Missing checkpoint: {GENERATOR_CKPT}")

saved  = torch.load(GENERATOR_CKPT, map_location=DEVICE, weights_only=False)
state  = saved["model_state_dict"]
BASE_CHANNELS = state["e1.weight"].shape[0]
EMBED_DIM     = state["style_embed.weight"].shape[1]

G = UNetGenerator(base_channels=BASE_CHANNELS, embed_dim=EMBED_DIM).to(DEVICE)
ckpt_meta = load_checkpoint(GENERATOR_CKPT, G, device=DEVICE)
G.eval()
print(f"Loaded generator from epoch {ckpt_meta.get('epoch')}, "
      f"val_loss={ckpt_meta.get('val_loss'):.4f}")


## 2. Test Set Evaluation: L1, PSNR, and SSIM per Style

In [ ]:
# 4. Run evaluation on the official FS2K test set
test_dataset = FS2KDataset(mode="test")
test_loader  = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=2)

style_sums  = defaultdict(lambda: defaultdict(float))
style_count = defaultdict(int)
records     = []   # for visualisation

with torch.inference_mode():
    for photo, sketch, style in test_loader:
        photo  = photo.to(DEVICE)
        sketch = sketch.to(DEVICE)
        style  = style.to(DEVICE)
        gen    = G(photo, style).clamp(-1, 1)

        # Shift [-1,1] -> [0,1] for metrics
        gen_01   = (gen + 1) * 0.5
        sketch_01 = (sketch + 1) * 0.5

        for i in range(len(photo)):
            s = style[i].item()
            l1  = F.l1_loss(gen_01[i:i+1], sketch_01[i:i+1]).item()
            ps  = psnr(gen_01[i:i+1], sketch_01[i:i+1])
            ssim_val = compute_ssim(gen_01[i:i+1], sketch_01[i:i+1]).item()

            style_count[s] += 1
            style_sums[s]["l1"]   += l1
            style_sums[s]["psnr"] += ps
            style_sums[s]["ssim"] += ssim_val

            if len(records) < 15:
                records.append({
                    "photo":  photo[i].cpu(),
                    "sketch": sketch[i].cpu(),
                    "gen":    gen[i].cpu(),
                    "style":  s,
                    "l1": l1, "psnr": ps, "ssim": ssim_val,
                })

# Summary table
rows = [{"style": STYLE_NAMES[s], "n": n,
         **{m: round(v / n, 4) for m, v in style_sums[s].items()}}
        for s, n in sorted(style_count.items())]
results_df = pd.DataFrame(rows)
display(results_df)


## 3. Style Conditioning Matrix (Photo × 3 Styles)

In [ ]:
# 5. Show same photo rendered into all 3 styles
n_photos = min(4, len(test_dataset))
sample_indices = list(range(n_photos))

fig, axes = plt.subplots(n_photos, 4, figsize=(14, 3.5 * n_photos), squeeze=False)

with torch.inference_mode():
    for row_i, idx in enumerate(sample_indices):
        photo_tensor, sketch_tensor, orig_style = test_dataset[idx]
        photo_in = photo_tensor.unsqueeze(0).to(DEVICE)

        photo_disp = ((photo_tensor.permute(1, 2, 0).numpy() + 1) * 0.5).clip(0, 1)
        axes[row_i, 0].imshow(photo_disp)
        axes[row_i, 0].set_title("Input Photo")
        axes[row_i, 0].axis("off")

        for col, style_id in enumerate([0, 1, 2]):
            style_tensor = torch.tensor([style_id], device=DEVICE)
            gen_sketch   = G(photo_in, style_tensor)
            gen_disp     = ((gen_sketch[0].cpu().permute(1, 2, 0).numpy() + 1) * 0.5).clip(0, 1)
            axes[row_i, col + 1].imshow(gen_disp)
            axes[row_i, col + 1].set_title(STYLE_NAMES[style_id])
            axes[row_i, col + 1].axis("off")

plt.suptitle("Style Conditioning Matrix: one photo → 3 sketch styles", y=1.01)
plt.tight_layout()
plt.show()


## 4. Failure Mode Analysis (Style Bleeding & Artefacts)

In [ ]:
# 6. Find >= 4 highest-error generated sketches
records.sort(key=lambda r: r["psnr"])   # lowest PSNR = worst
failures = records[:4]

fig, axes = plt.subplots(len(failures), 3, figsize=(11, 3.2 * len(failures)), squeeze=False)
for row, rec in zip(axes, failures):
    photo_disp  = ((rec["photo"].permute(1, 2, 0).numpy()  + 1) * 0.5).clip(0, 1)
    sketch_disp = ((rec["sketch"].permute(1, 2, 0).numpy() + 1) * 0.5).clip(0, 1)
    gen_disp    = ((rec["gen"].permute(1, 2, 0).numpy()    + 1) * 0.5).clip(0, 1)

    for ax, image, title in zip(row, [photo_disp, gen_disp, sketch_disp],
                                 ["Input Photo", "Generated Sketch", "Ground Truth"]):
        ax.imshow(image)
        ax.set_title(title, fontsize=9)
        ax.axis("off")

    print(f"Style {rec['style']} | L1={rec['l1']:.4f} | "
          f"PSNR={rec['psnr']:.2f} dB | SSIM={rec['ssim']:.3f}")
    print("  Possible causes: style bleeding (textures from adjacent style), "
          "missing facial detail (glasses, beard), or blurry edges in hair region.")

plt.tight_layout()
plt.show()